# 🏆 FTMO Strategy Tester & Optimizer

A rigorous prop-firm focused backtesting framework with:
- **FTMO Rule Enforcement** (5% daily DD, 10% max DD, 10% profit target)
- **Walk-Forward Optimization** (avoids in-sample overfitting)
- **Purged K-Fold CV** (prevents data leakage)
- **Optuna Grid/Bayesian Search** (efficient hyperparameter tuning)
- **Overfitting Detection** (IS vs OOS Sharpe degradation analysis)
- **Full Performance Dashboard** (equity curve, drawdown, heatmaps)

---
> ⚠️ **Honest caveat**: No framework can *guarantee* a Sharpe ≥ 1.0 OOS.
> Even Robert Carver notes that highly diversified systematic systems are *unlikely* to exceed SR 1.0.
> This notebook maximises your chances through rigorous methodology — not curve fitting.
---

## 0. Install & Import Dependencies

In [ ]:
import subprocess, sys
pkgs = ['yfinance', 'optuna', 'seaborn', 'matplotlib', 'scipy', 'tqdm']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages installed (no pandas-ta — pure numpy indicators)')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
import optuna
from scipy import stats
from itertools import product
from tqdm.notebook import tqdm
import json, warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Callable

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('dark_background')
sns.set_palette('husl')

FTMO_GREEN = '#00FF88'
FTMO_RED   = '#FF4466'
FTMO_BLUE  = '#4488FF'
FTMO_GOLD  = '#FFD700'

print('✅ Imports complete')

## 1. ⚙️ Configuration — Edit Everything Here

In [ ]:
# ============================================================
#   FTMO CHALLENGE SETTINGS
# ============================================================
FTMO_CONFIG = {
    'account_size'       : 100_000,   # Challenge account size ($)
    'profit_target_pct'  : 0.10,      # 10% profit target
    'max_daily_dd_pct'   : 0.05,      # 5%  max daily drawdown
    'max_total_dd_pct'   : 0.10,      # 10% max total drawdown
    'max_trading_days'   : 30,        # Phase 1: 30 calendar days
    'min_trading_days'   : 4,         # Must trade at least 4 days
}

# ============================================================
#   DATA SETTINGS
# ============================================================
DATA_CONFIG = {
    'ticker'    : 'EURUSD=X',         # Ticker (yfinance format)
    'interval'  : '1h',               # '1m','5m','15m','1h','1d'
    'start'     : '2018-01-01',
    'end'       : '2024-12-31',
    'spread_pips': 1.0,               # Spread cost in pips (1 pip = 0.0001 for FX)
    'commission' : 0.0,               # Commission per trade ($)
    'slippage_pips': 0.5,             # Slippage in pips
}

# ============================================================
#   OPTIMIZATION SETTINGS
# ============================================================
OPT_CONFIG = {
    'n_trials'          : 200,        # Optuna trials per fold
    'n_folds'           : 5,          # Walk-forward folds
    'train_pct'         : 0.70,       # 70% train, 30% test per fold
    'purge_bars'        : 24,         # Bars to purge between train/test (avoid leakage)
    'embargo_bars'      : 12,         # Embargo period after test
    'objective'         : 'sharpe',   # 'sharpe', 'sortino', 'calmar', 'profit_factor'
    'min_trades'        : 30,         # Minimum trades per fold to avoid degenerate results
    'seed'              : 42,
}

print('✅ Configuration set')
print(f"   Account: ${FTMO_CONFIG['account_size']:,}")
print(f"   Profit target: {FTMO_CONFIG['profit_target_pct']*100:.0f}%  |  "
      f"Max Daily DD: {FTMO_CONFIG['max_daily_dd_pct']*100:.0f}%  |  "
      f"Max Total DD: {FTMO_CONFIG['max_total_dd_pct']*100:.0f}%")
print(f"   Ticker: {DATA_CONFIG['ticker']} @ {DATA_CONFIG['interval']}")

## 2. 📥 Data Loading & Preprocessing

In [ ]:
def load_data(ticker: str, interval: str, start: str, end: str) -> pd.DataFrame:
    """Download OHLCV data and compute base features."""
    print(f'📥 Downloading {ticker} [{interval}] {start} → {end}...')
    df = yf.download(ticker, start=start, end=end, interval=interval,
                     auto_adjust=True, progress=False)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df.dropna(inplace=True)
    print(f'   ✅ {len(df):,} bars loaded  |  {df.index[0].date()} → {df.index[-1].date()}')
    return df

df_raw = load_data(
    DATA_CONFIG['ticker'],
    DATA_CONFIG['interval'],
    DATA_CONFIG['start'],
    DATA_CONFIG['end']
)
df_raw.tail(3)

## 3. 📊 Core Performance Engine

In [ ]:
class PerformanceEngine:
    """Compute all trading metrics including FTMO-specific ones."""

    @staticmethod
    def compute_metrics(equity: pd.Series, trades: list,
                        account_size: float = 100_000,
                        periods_per_year: int = 252) -> Dict:
        if equity is None or len(equity) < 5:
            return {}

        returns = equity.pct_change().dropna()
        if returns.std() == 0:
            return {}

        total_return    = (equity.iloc[-1] / equity.iloc[0]) - 1
        n_years         = len(returns) / periods_per_year
        cagr            = (1 + total_return) ** (1 / max(n_years, 0.01)) - 1

        sharpe          = returns.mean() / returns.std() * np.sqrt(periods_per_year)
        downside        = returns[returns < 0].std()
        sortino         = returns.mean() / downside * np.sqrt(periods_per_year) if downside > 0 else 0

        roll_max        = equity.cummax()
        dd_series       = (equity - roll_max) / roll_max
        max_dd          = dd_series.min()
        calmar          = cagr / abs(max_dd) if max_dd != 0 else 0

        # DD duration
        in_dd           = (dd_series < 0).astype(int)
        dd_groups       = (in_dd.diff() != 0).cumsum()
        dd_durations    = in_dd.groupby(dd_groups).sum()
        max_dd_dur      = int(dd_durations.max()) if len(dd_durations) > 0 else 0

        # Var / CVaR
        var_95          = float(np.percentile(returns, 5))
        cvar_95         = float(returns[returns <= var_95].mean())

        # Trade stats
        if trades and len(trades) > 0:
            pnls        = [t['pnl'] for t in trades]
            wins        = [p for p in pnls if p > 0]
            losses      = [p for p in pnls if p <= 0]
            win_rate    = len(wins) / len(pnls)
            avg_win     = np.mean(wins)   if wins   else 0
            avg_loss    = np.mean(losses) if losses else 0
            profit_factor = (sum(wins) / abs(sum(losses))
                             if losses and sum(losses) != 0 else np.inf)
            expectancy  = win_rate * avg_win + (1 - win_rate) * avg_loss
        else:
            win_rate = avg_win = avg_loss = profit_factor = expectancy = 0

        # FTMO compliance
        daily_equity    = equity.resample('1D').last().dropna()
        daily_dd        = (daily_equity - daily_equity.cummax()) / account_size
        max_daily_dd    = daily_dd.min()
        ftmo_daily_ok   = max_daily_dd >= -0.05
        ftmo_total_ok   = max_dd >= -0.10
        ftmo_profit_ok  = total_return >= 0.10

        return {
            'Total Return %'    : round(total_return * 100, 2),
            'CAGR %'            : round(cagr * 100, 2),
            'Sharpe'            : round(sharpe, 3),
            'Sortino'           : round(sortino, 3),
            'Calmar'            : round(calmar, 3),
            'Max Drawdown %'    : round(max_dd * 100, 2),
            'Max DD Duration'   : max_dd_dur,
            'VaR 95%'           : round(var_95 * 100, 3),
            'CVaR 95%'          : round(cvar_95 * 100, 3),
            'Trades'            : len(trades) if trades else 0,
            'Win Rate %'        : round(win_rate * 100, 1),
            'Avg Win'           : round(avg_win, 2),
            'Avg Loss'          : round(avg_loss, 2),
            'Profit Factor'     : round(profit_factor, 3),
            'Expectancy'        : round(expectancy, 2),
            'FTMO Daily DD OK'  : ftmo_daily_ok,
            'FTMO Total DD OK'  : ftmo_total_ok,
            'FTMO Profit OK'    : ftmo_profit_ok,
            'FTMO PASS'         : (ftmo_daily_ok and ftmo_total_ok and ftmo_profit_ok),
        }

print('✅ PerformanceEngine defined')

## 4. 🏗️ Strategy Definitions
Define your strategies below. Each strategy receives a DataFrame and a params dict, and returns `(signals, df_with_indicators)`.

In [ ]:
# ─────────────────────────────────────────
#  PURE NUMPY/PANDAS INDICATOR LIBRARY
#  (no external ta library needed)
# ─────────────────────────────────────────

def _atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def _rsi(close, period=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / loss.replace(0, 1e-9)
    return 100 - 100 / (1 + rs)

def _bbands(close, period=20, std_mult=2.0):
    ma    = close.rolling(period).mean()
    sigma = close.rolling(period).std()
    return ma + std_mult * sigma, ma - std_mult * sigma

def _macd(close, fast=12, slow=26, signal=9):
    ema_f  = close.ewm(span=fast,   adjust=False).mean()
    ema_s  = close.ewm(span=slow,   adjust=False).mean()
    macd   = ema_f - ema_s
    sig    = macd.ewm(span=signal, adjust=False).mean()
    return macd, sig

def _adx(high, low, close, period=14):
    up   = high.diff()
    down = -low.diff()
    pos_dm = up.where((up > down) & (up > 0), 0.0)
    neg_dm = down.where((down > up) & (down > 0), 0.0)
    atr_v  = _atr(high, low, close, period)
    smooth = lambda s: s.ewm(alpha=1/period, adjust=False).mean()
    pdi    = 100 * smooth(pos_dm) / atr_v.replace(0, 1e-9)
    ndi    = 100 * smooth(neg_dm) / atr_v.replace(0, 1e-9)
    dx     = 100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, 1e-9)
    return dx.ewm(alpha=1/period, adjust=False).mean()


# ─────────────────────────────────────────
#  STRATEGY LIBRARY
# ─────────────────────────────────────────

def strategy_ema_cross(df: pd.DataFrame, params: dict) -> pd.Series:
    fast       = params.get('fast', 9)
    slow       = params.get('slow', 21)
    atr_period = params.get('atr_period', 14)
    atr_mult   = params.get('atr_mult', 1.0)
    ema_f = df['Close'].ewm(span=fast, adjust=False).mean()
    ema_s = df['Close'].ewm(span=slow, adjust=False).mean()
    atr   = _atr(df['High'], df['Low'], df['Close'], atr_period)
    atr_ma = atr.rolling(50).mean()
    raw_signal = np.where(ema_f > ema_s, 1, -1)
    vol_filter = (atr > atr_ma * atr_mult).values
    return pd.Series(raw_signal * vol_filter, index=df.index)


def strategy_rsi_mean_revert(df: pd.DataFrame, params: dict) -> pd.Series:
    rsi_period = params.get('rsi_period', 14)
    oversold   = params.get('oversold', 30)
    overbought = params.get('overbought', 70)
    ema_trend  = params.get('ema_trend', 200)
    rsi   = _rsi(df['Close'], rsi_period)
    trend = df['Close'].ewm(span=ema_trend, adjust=False).mean()
    signal = pd.Series(0, index=df.index)
    signal[(rsi < oversold)  & (df['Close'] > trend)] =  1
    signal[(rsi > overbought) & (df['Close'] < trend)] = -1
    return signal


def strategy_bb_breakout(df: pd.DataFrame, params: dict) -> pd.Series:
    bb_period  = params.get('bb_period', 20)
    bb_std     = params.get('bb_std', 2.0)
    rsi_period = params.get('rsi_period', 14)
    rsi_thresh = params.get('rsi_thresh', 50)
    upper, lower = _bbands(df['Close'], bb_period, bb_std)
    rsi = _rsi(df['Close'], rsi_period)
    signal = pd.Series(0, index=df.index)
    signal[(df['Close'] > upper) & (rsi > rsi_thresh)] =  1
    signal[(df['Close'] < lower) & (rsi < rsi_thresh)] = -1
    return signal


def strategy_macd_trend(df: pd.DataFrame, params: dict) -> pd.Series:
    fast       = params.get('fast', 12)
    slow       = params.get('slow', 26)
    signal_p   = params.get('signal', 9)
    adx_period = params.get('adx_period', 14)
    adx_thresh = params.get('adx_thresh', 25)
    macd_line, signal_line = _macd(df['Close'], fast, slow, signal_p)
    adx_val = _adx(df['High'], df['Low'], df['Close'], adx_period)
    trend_strong = adx_val > adx_thresh
    signal = pd.Series(0, index=df.index)
    signal[(macd_line > signal_line) & trend_strong] =  1
    signal[(macd_line < signal_line) & trend_strong] = -1
    return signal


def strategy_donchian_breakout(df: pd.DataFrame, params: dict) -> pd.Series:
    period      = params.get('period', 20)
    exit_period = params.get('exit_period', 10)
    upper = df['High'].rolling(period).max()
    lower = df['Low'].rolling(period).min()
    exit_upper = df['High'].rolling(exit_period).max()
    exit_lower = df['Low'].rolling(exit_period).min()
    signal = pd.Series(0, index=df.index)
    position = 0
    for i in range(period, len(df)):
        price = df['Close'].iloc[i]
        if position == 0:
            if price >= upper.iloc[i-1]:  position = 1
            elif price <= lower.iloc[i-1]: position = -1
        elif position == 1:
            if price <= exit_lower.iloc[i-1]: position = 0
        elif position == -1:
            if price >= exit_upper.iloc[i-1]: position = 0
        signal.iloc[i] = position
    return signal


STRATEGY_REGISTRY = {
    'EMA Cross'         : strategy_ema_cross,
    'RSI Mean Revert'   : strategy_rsi_mean_revert,
    'BB Breakout'       : strategy_bb_breakout,
    'MACD Trend'        : strategy_macd_trend,
    'Donchian Breakout' : strategy_donchian_breakout,
}

PARAM_SPACES = {
    'EMA Cross': {
        'fast'       : ('int',   5,  50),
        'slow'       : ('int',  20, 200),
        'atr_period' : ('int',   7,  21),
        'atr_mult'   : ('float', 0.5, 2.0),
    },
    'RSI Mean Revert': {
        'rsi_period' : ('int',   7,  21),
        'oversold'   : ('int',  20,  40),
        'overbought' : ('int',  60,  80),
        'ema_trend'  : ('int', 100, 300),
    },
    'BB Breakout': {
        'bb_period'  : ('int',  10,  50),
        'bb_std'     : ('float', 1.5, 3.0),
        'rsi_period' : ('int',   7,  21),
        'rsi_thresh' : ('int',  40,  60),
    },
    'MACD Trend': {
        'fast'       : ('int',   8,  20),
        'slow'       : ('int',  20,  50),
        'signal'     : ('int',   5,  15),
        'adx_period' : ('int',   7,  21),
        'adx_thresh' : ('int',  15,  35),
    },
    'Donchian Breakout': {
        'period'      : ('int',  10,  60),
        'exit_period' : ('int',   5,  30),
    },
}

print(f'✅ {len(STRATEGY_REGISTRY)} strategies registered (pure numpy — no pandas-ta): {list(STRATEGY_REGISTRY.keys())}')


## 5. 🔄 Backtesting Engine with FTMO Rules

In [ ]:
class FTMOBacktester:
    """Vectorized backtester with full FTMO rule enforcement."""

    def __init__(self, ftmo_config: dict, data_config: dict):
        self.ftmo   = ftmo_config
        self.data   = data_config

    def _compute_pip_cost(self, df: pd.DataFrame) -> float:
        """Estimate pip cost from asset price."""
        price = df['Close'].mean()
        if price < 10:       # FX major
            return 0.0001
        elif price < 500:    # Indices
            return 0.01
        else:                # Crypto/commodities
            return price * 0.0001

    def run(self, df: pd.DataFrame, signal: pd.Series,
            position_size: float = 0.02,
            stop_loss_pct: float = 0.01,
            take_profit_pct: float = 0.02,
            use_trailing_stop: bool = False) -> Tuple[pd.Series, List]:
        """
        Run backtest with event-driven stop/TP and FTMO enforcement.

        Returns:
            equity  : pd.Series of account equity
            trades  : list of trade dicts
        """
        account    = self.ftmo['account_size']
        equity     = [account]
        equity_idx = [df.index[0]]
        trades     = []

        pip_size   = self._compute_pip_cost(df)
        spread_cost = (self.data['spread_pips'] + self.data['slippage_pips']) * pip_size

        daily_start_equity = account
        last_day           = df.index[0].date()
        peak_equity        = account
        ftmo_breached      = False

        position  = 0       # -1, 0, 1
        entry_px  = 0.0
        trail_px  = 0.0
        current_equity = account

        sig = signal.reindex(df.index).fillna(0)

        for i in range(1, len(df)):
            bar      = df.iloc[i]
            prev_bar = df.iloc[i - 1]
            curr_date = df.index[i].date()

            # Reset daily drawdown tracker
            if curr_date != last_day:
                daily_start_equity = current_equity
                last_day = curr_date

            # Check FTMO breach
            daily_dd = (current_equity - daily_start_equity) / self.ftmo['account_size']
            total_dd = (current_equity - peak_equity) / peak_equity

            if daily_dd <= -self.ftmo['max_daily_dd_pct'] or \
               total_dd <= -self.ftmo['max_total_dd_pct']:
                ftmo_breached = True
                if position != 0:
                    # Force close
                    close_px = bar['Open']
                    pnl = position * (close_px - entry_px) / entry_px * position_size * current_equity
                    pnl -= spread_cost * position_size * current_equity
                    current_equity = max(current_equity + pnl, 0)
                    trades.append({'entry': entry_px, 'exit': close_px,
                                   'pnl': pnl, 'direction': position,
                                   'forced_close': True})
                    position = 0
                break

            close_px = bar['Close']
            new_signal = int(sig.iloc[i])

            # Exit existing position
            if position != 0:
                # Stop loss check
                sl_triggered = False
                tp_triggered = False

                if position == 1:
                    sl_px = entry_px * (1 - stop_loss_pct)
                    tp_px = entry_px * (1 + take_profit_pct)
                    if use_trailing_stop:
                        trail_px = max(trail_px, bar['High'] * (1 - stop_loss_pct))
                        sl_px = trail_px
                    if bar['Low'] <= sl_px:
                        sl_triggered = True
                        exit_px = sl_px
                    elif bar['High'] >= tp_px:
                        tp_triggered = True
                        exit_px = tp_px
                else:  # short
                    sl_px = entry_px * (1 + stop_loss_pct)
                    tp_px = entry_px * (1 - take_profit_pct)
                    if use_trailing_stop:
                        trail_px = min(trail_px, bar['Low'] * (1 + stop_loss_pct))
                        sl_px = trail_px
                    if bar['High'] >= sl_px:
                        sl_triggered = True
                        exit_px = sl_px
                    elif bar['Low'] <= tp_px:
                        tp_triggered = True
                        exit_px = tp_px

                if sl_triggered or tp_triggered or (new_signal != position and new_signal != 0):
                    exit_px = exit_px if (sl_triggered or tp_triggered) else close_px
                    pnl = position * (exit_px - entry_px) / entry_px * position_size * current_equity
                    pnl -= spread_cost * position_size * current_equity
                    pnl -= self.data['commission']
                    current_equity += pnl
                    trades.append({
                        'entry': entry_px, 'exit': exit_px, 'pnl': pnl,
                        'direction': position,
                        'sl': sl_triggered, 'tp': tp_triggered,
                        'bar_idx': i
                    })
                    position = 0

            # Enter new position
            if position == 0 and new_signal != 0 and not ftmo_breached:
                position = new_signal
                entry_px = close_px + (spread_cost * new_signal)
                trail_px = entry_px * (1 - stop_loss_pct) if new_signal == 1 else entry_px * (1 + stop_loss_pct)

            # Mark-to-market unrealised P&L
            if position != 0:
                unreal = position * (close_px - entry_px) / entry_px * position_size * current_equity
                displayed_equity = current_equity + unreal
            else:
                displayed_equity = current_equity

            peak_equity = max(peak_equity, displayed_equity)
            equity.append(displayed_equity)
            equity_idx.append(df.index[i])

        equity_series = pd.Series(equity, index=equity_idx)
        return equity_series, trades

backtester = FTMOBacktester(FTMO_CONFIG, DATA_CONFIG)
print('✅ FTMOBacktester ready')

## 6. 🧮 Walk-Forward Optimizer with Purged K-Fold

In [ ]:
class WalkForwardOptimizer:
    """Walk-forward optimization with purged embargo and Optuna."""

    def __init__(self, strategy_fn: Callable, param_space: dict,
                 opt_config: dict, backtester: FTMOBacktester):
        self.strategy_fn = strategy_fn
        self.param_space = param_space
        self.opt         = opt_config
        self.bt          = backtester

    def _make_folds(self, df: pd.DataFrame) -> List[Tuple]:
        """Create purged walk-forward folds."""
        n        = len(df)
        n_folds  = self.opt['n_folds']
        purge    = self.opt['purge_bars']
        embargo  = self.opt['embargo_bars']
        fold_size = n // (n_folds + 1)

        folds = []
        for k in range(n_folds):
            train_end  = fold_size * (k + 1)
            test_start = train_end + purge
            test_end   = min(test_start + fold_size, n - embargo)
            if test_end > test_start:
                folds.append((
                    df.iloc[:train_end],
                    df.iloc[test_start:test_end]
                ))
        return folds

    def _objective(self, trial, df_train: pd.DataFrame,
                   bt_params: dict) -> float:
        params = {}
        for name, spec in self.param_space.items():
            typ = spec[0]
            if typ == 'int':
                params[name] = trial.suggest_int(name, spec[1], spec[2])
            elif typ == 'float':
                params[name] = trial.suggest_float(name, spec[1], spec[2])
            elif typ == 'categorical':
                params[name] = trial.suggest_categorical(name, spec[1])

        # Constraint: fast < slow for EMA/MACD
        if 'fast' in params and 'slow' in params:
            if params['fast'] >= params['slow']:
                return -1e6

        try:
            sig    = self.strategy_fn(df_train, params)
            equity, trades = self.bt.run(df_train, sig, **bt_params)
        except Exception:
            return -1e6

        if len(trades) < self.opt['min_trades']:
            return -1e6

        metrics = PerformanceEngine.compute_metrics(
            equity, trades, self.bt.ftmo['account_size'])
        if not metrics:
            return -1e6

        obj = self.opt['objective'].lower()
        mapping = {
            'sharpe'        : metrics.get('Sharpe', -1e6),
            'sortino'       : metrics.get('Sortino', -1e6),
            'calmar'        : metrics.get('Calmar', -1e6),
            'profit_factor' : metrics.get('Profit Factor', -1e6),
        }
        return float(mapping.get(obj, metrics.get('Sharpe', -1e6)))

    def optimize(self, df: pd.DataFrame, bt_params: dict,
                 verbose: bool = True) -> dict:
        """
        Run walk-forward optimization.

        Returns dict with IS/OOS results per fold and aggregate stats.
        """
        folds      = self._make_folds(df)
        results    = []
        all_oos_eq = []
        all_oos_trades = []

        print(f'\n🔄 Walk-Forward Optimization — {len(folds)} folds')
        print(f'   Strategy: {self.strategy_fn.__name__}')
        print(f'   Objective: {self.opt["objective"]} | Trials: {self.opt["n_trials"]}\n')

        for k, (df_train, df_test) in enumerate(folds):
            print(f'  Fold {k+1}/{len(folds)} — '
                  f'Train: {df_train.index[0].date()}→{df_train.index[-1].date()} '
                  f'({len(df_train):,} bars)  '
                  f'Test: {df_test.index[0].date()}→{df_test.index[-1].date()} '
                  f'({len(df_test):,} bars)')

            # Optuna optimization
            study = optuna.create_study(
                direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=self.opt['seed'] + k)
            )
            study.optimize(
                lambda trial: self._objective(trial, df_train, bt_params),
                n_trials=self.opt['n_trials'],
                show_progress_bar=False
            )

            best_params = study.best_params
            best_is_val = study.best_value

            # IS performance
            sig_is  = self.strategy_fn(df_train, best_params)
            eq_is, trades_is = self.bt.run(df_train, sig_is, **bt_params)
            m_is    = PerformanceEngine.compute_metrics(
                eq_is, trades_is, self.bt.ftmo['account_size'])

            # OOS performance
            sig_oos = self.strategy_fn(df_test, best_params)
            eq_oos, trades_oos = self.bt.run(df_test, sig_oos, **bt_params)
            m_oos   = PerformanceEngine.compute_metrics(
                eq_oos, trades_oos, self.bt.ftmo['account_size'])

            is_sharpe  = m_is.get('Sharpe', 0)
            oos_sharpe = m_oos.get('Sharpe', 0)
            ratio      = oos_sharpe / is_sharpe if is_sharpe != 0 else 0
            degrade    = '🟢' if ratio >= 0.6 else '🟡' if ratio >= 0.3 else '🔴'

            print(f'     IS Sharpe: {is_sharpe:+.3f}  |  '
                  f'OOS Sharpe: {oos_sharpe:+.3f}  |  '
                  f'OOS/IS ratio: {ratio:.2f} {degrade}  |  '
                  f'Trades: {m_oos.get("Trades", 0)}')

            all_oos_eq.append(eq_oos)
            all_oos_trades.extend(trades_oos)
            results.append({
                'fold'          : k + 1,
                'best_params'   : best_params,
                'is_metrics'    : m_is,
                'oos_metrics'   : m_oos,
                'is_sharpe'     : is_sharpe,
                'oos_sharpe'    : oos_sharpe,
                'oos_is_ratio'  : ratio,
                'eq_oos'        : eq_oos,
                'trades_oos'    : trades_oos,
                'study'         : study,
            })

        # Combined OOS equity (re-scale each fold to start from prior fold end)
        combined_equity = self._stitch_equity(all_oos_eq)

        agg_metrics = PerformanceEngine.compute_metrics(
            combined_equity, all_oos_trades, self.bt.ftmo['account_size'])

        avg_oos_sharpe = np.mean([r['oos_sharpe'] for r in results])
        avg_ratio      = np.mean([r['oos_is_ratio'] for r in results])

        print(f'\n{'='*60}')
        print(f'📊 AGGREGATE OOS RESULTS')
        print(f'   Avg OOS Sharpe : {avg_oos_sharpe:.3f}')
        print(f'   Avg OOS/IS     : {avg_ratio:.2f}')
        print(f'   Combined Sharpe: {agg_metrics.get("Sharpe", 0):.3f}')
        print(f'   Max DD         : {agg_metrics.get("Max Drawdown %", 0):.2f}%')
        print(f'   Win Rate       : {agg_metrics.get("Win Rate %", 0):.1f}%')
        print(f'{'='*60}\n')

        return {
            'fold_results'    : results,
            'combined_equity' : combined_equity,
            'all_trades'      : all_oos_trades,
            'agg_metrics'     : agg_metrics,
            'avg_oos_sharpe'  : avg_oos_sharpe,
            'avg_oos_is_ratio': avg_ratio,
        }

    @staticmethod
    def _stitch_equity(eq_list: List[pd.Series]) -> pd.Series:
        """Chain equity curves together at natural join points."""
        result = []
        offset = 0.0
        for eq in eq_list:
            if len(eq) == 0:
                continue
            if len(result) == 0:
                result.append(eq)
                offset = eq.iloc[-1] - eq.iloc[0]
            else:
                scaled = eq - eq.iloc[0] + result[-1].iloc[-1]
                result.append(scaled)
        if not result:
            return pd.Series(dtype=float)
        return pd.concat(result).sort_index()

print('✅ WalkForwardOptimizer defined')

## 7. 🎯 Run Optimization
Select your strategy and run the full walk-forward optimization.

In [ ]:
# ============================================================
#   SELECT YOUR STRATEGY HERE
# ============================================================
ACTIVE_STRATEGY = 'EMA Cross'   # Change to any key in STRATEGY_REGISTRY

# ============================================================
#   EXECUTION / RISK PARAMS (adjust per strategy)
# ============================================================
BT_PARAMS = {
    'position_size'      : 0.02,    # Risk 2% of account per trade
    'stop_loss_pct'      : 0.01,    # 1% stop loss
    'take_profit_pct'    : 0.02,    # 2% take profit (1:2 RR)
    'use_trailing_stop'  : False,   # Toggle trailing stop
}

# ============================================================
#   RUN
# ============================================================
strategy_fn = STRATEGY_REGISTRY[ACTIVE_STRATEGY]
param_space  = PARAM_SPACES[ACTIVE_STRATEGY]

optimizer = WalkForwardOptimizer(
    strategy_fn = strategy_fn,
    param_space  = param_space,
    opt_config   = OPT_CONFIG,
    backtester   = backtester
)

wf_results = optimizer.optimize(df_raw, BT_PARAMS, verbose=True)

## 8. 📈 Performance Dashboard

In [ ]:
def plot_dashboard(wf_results: dict, strategy_name: str, ftmo_config: dict):
    """Full performance dashboard."""
    combined_eq = wf_results['combined_equity']
    metrics     = wf_results['agg_metrics']
    fold_results = wf_results['fold_results']
    trades      = wf_results['all_trades']

    fig = plt.figure(figsize=(22, 26), facecolor='#0d1117')
    gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.45, wspace=0.35)

    title_color = '#E8E8E8'
    grid_color  = '#222233'

    # ── Title ──
    fig.suptitle(
        f'FTMO Strategy Dashboard  ·  {strategy_name}',
        fontsize=20, color=title_color, fontweight='bold', y=0.98
    )

    # ── 1. Equity Curve ──
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor('#0d1117')
    ax1.plot(combined_eq, color=FTMO_GREEN, lw=1.5, label='Portfolio Equity')
    ax1.fill_between(combined_eq.index, combined_eq, combined_eq.iloc[0],
                     alpha=0.12, color=FTMO_GREEN)
    target_line = ftmo_config['account_size'] * (1 + ftmo_config['profit_target_pct'])
    ax1.axhline(target_line, color=FTMO_GOLD,  ls='--', lw=1.2, alpha=0.8, label='10% Profit Target')
    ax1.axhline(ftmo_config['account_size'] * 0.90, color=FTMO_RED, ls='--', lw=1.2, alpha=0.8, label='10% Max DD Limit')
    # Fold separators
    for r in fold_results:
        eq = r['eq_oos']
        if len(eq) > 0:
            ax1.axvline(eq.index[0], color='#555577', ls=':', lw=0.8, alpha=0.5)
    ax1.set_title('Stitched OOS Equity Curve', color=title_color, fontsize=13)
    ax1.set_ylabel('Account Value ($)', color=title_color)
    ax1.tick_params(colors=title_color)
    ax1.grid(True, color=grid_color, alpha=0.5)
    ax1.legend(loc='upper left', fontsize=9)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # ── 2. Drawdown ──
    ax2 = fig.add_subplot(gs[1, :])
    ax2.set_facecolor('#0d1117')
    roll_max = combined_eq.cummax()
    dd_series = (combined_eq - roll_max) / roll_max * 100
    ax2.fill_between(dd_series.index, dd_series, 0, alpha=0.7, color=FTMO_RED)
    ax2.axhline(-5,  color=FTMO_GOLD, ls='--', lw=1, alpha=0.8, label='Daily DD Limit (-5%)')
    ax2.axhline(-10, color='#FF0000', ls='--', lw=1.2, alpha=0.9, label='Total DD Limit (-10%)')
    ax2.set_title('Drawdown %', color=title_color, fontsize=13)
    ax2.set_ylabel('Drawdown (%)', color=title_color)
    ax2.tick_params(colors=title_color)
    ax2.grid(True, color=grid_color, alpha=0.5)
    ax2.legend(loc='lower left', fontsize=9)

    # ── 3. IS vs OOS Sharpe ──
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.set_facecolor('#0d1117')
    fold_nums = [r['fold'] for r in fold_results]
    is_sharpes  = [r['is_sharpe']  for r in fold_results]
    oos_sharpes = [r['oos_sharpe'] for r in fold_results]
    x  = np.arange(len(fold_nums))
    bw = 0.35
    ax3.bar(x - bw/2, is_sharpes,  bw, label='IS Sharpe',  color=FTMO_BLUE,  alpha=0.8)
    ax3.bar(x + bw/2, oos_sharpes, bw, label='OOS Sharpe', color=FTMO_GREEN, alpha=0.8)
    ax3.axhline(1.0, color=FTMO_GOLD, ls='--', lw=1.5, label='SR = 1.0 Target')
    ax3.axhline(0.0, color='white', ls='-',  lw=0.5, alpha=0.3)
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'Fold {k}' for k in fold_nums])
    ax3.set_title('IS vs OOS Sharpe Ratio', color=title_color, fontsize=11)
    ax3.legend(fontsize=8)
    ax3.tick_params(colors=title_color)
    ax3.grid(True, color=grid_color, alpha=0.4, axis='y')

    # ── 4. OOS/IS Degradation ──
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.set_facecolor('#0d1117')
    ratios = [r['oos_is_ratio'] for r in fold_results]
    colors_r = [FTMO_GREEN if r >= 0.6 else FTMO_GOLD if r >= 0.3 else FTMO_RED for r in ratios]
    ax4.bar(fold_nums, ratios, color=colors_r, alpha=0.85)
    ax4.axhline(0.6, color=FTMO_GREEN, ls='--', lw=1.2, label='Good (≥0.6)')
    ax4.axhline(0.3, color=FTMO_GOLD,  ls='--', lw=1.2, label='Marginal (≥0.3)')
    ax4.set_title('OOS / IS Sharpe Ratio\n(Overfitting Detector)', color=title_color, fontsize=11)
    ax4.set_xlabel('Fold', color=title_color)
    ax4.legend(fontsize=8)
    ax4.tick_params(colors=title_color)
    ax4.grid(True, color=grid_color, alpha=0.4, axis='y')

    # ── 5. Trade PnL Distribution ──
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.set_facecolor('#0d1117')
    pnls = [t['pnl'] for t in trades if 'pnl' in t]
    if pnls:
        pos_pnls = [p for p in pnls if p >= 0]
        neg_pnls = [p for p in pnls if p < 0]
        ax5.hist(pos_pnls, bins=30, alpha=0.7, color=FTMO_GREEN, label='Winners')
        ax5.hist(neg_pnls, bins=30, alpha=0.7, color=FTMO_RED,   label='Losers')
        ax5.axvline(0, color='white', lw=1, alpha=0.5)
        ax5.axvline(np.mean(pnls), color=FTMO_GOLD, ls='--', lw=1.5,
                    label=f'Avg ${np.mean(pnls):.0f}')
    ax5.set_title('Trade P&L Distribution', color=title_color, fontsize=11)
    ax5.legend(fontsize=8)
    ax5.tick_params(colors=title_color)
    ax5.grid(True, color=grid_color, alpha=0.4)

    # ── 6. Monthly Returns Heatmap ──
    ax6 = fig.add_subplot(gs[3, :])
    ax6.set_facecolor('#0d1117')
    if len(combined_eq) > 30:
        monthly = combined_eq.resample('ME').last().pct_change() * 100
        monthly_df = monthly.to_frame('Return')
        monthly_df['Year']  = monthly_df.index.year
        monthly_df['Month'] = monthly_df.index.month
        pivot = monthly_df.pivot_table(values='Return', index='Year', columns='Month')
        pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
        sns.heatmap(
            pivot, ax=ax6, cmap='RdYlGn', center=0,
            annot=True, fmt='.1f', annot_kws={'size': 8},
            cbar_kws={'label': 'Return %'},
            linewidths=0.5, linecolor='#0d1117'
        )
        ax6.set_title('Monthly Returns Heatmap (%)', color=title_color, fontsize=11)
        ax6.tick_params(colors=title_color)

    # ── 7. Metrics Table ──
    ax7 = fig.add_subplot(gs[4, :])
    ax7.set_facecolor('#0d1117')
    ax7.axis('off')

    metric_keys = [
        'Sharpe', 'Sortino', 'Calmar',
        'Total Return %', 'CAGR %', 'Max Drawdown %',
        'Trades', 'Win Rate %', 'Profit Factor', 'Expectancy',
        'FTMO Daily DD OK', 'FTMO Total DD OK', 'FTMO Profit OK', 'FTMO PASS'
    ]
    rows = []
    for k in metric_keys:
        v = metrics.get(k, 'N/A')
        rows.append([k, str(v)])

    # Split into 3 columns
    n = len(rows)
    col_size = (n + 2) // 3
    col_data  = [rows[i:i+col_size] for i in range(0, n, col_size)]

    for ci, col in enumerate(col_data):
        col_labels = [r[0] for r in col]
        col_vals   = [r[1] for r in col]
        x_start    = 0.02 + ci * 0.34
        cell_h     = 0.12
        for ri, (lbl, val) in enumerate(zip(col_labels, col_vals)):
            y = 0.85 - ri * cell_h
            val_color = FTMO_GREEN if val in ('True', ) else \
                        FTMO_RED   if val in ('False',) else '#CCCCCC'
            ax7.text(x_start,      y, lbl, transform=ax7.transAxes,
                     color='#AAAACC', fontsize=9.5, va='top')
            ax7.text(x_start+0.18, y, val, transform=ax7.transAxes,
                     color=val_color, fontsize=9.5, va='top', fontweight='bold')
    ax7.set_title('Aggregate OOS Metrics', color=title_color, fontsize=11, pad=10)

    plt.savefig('/tmp/ftmo_dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    print('💾 Dashboard saved to /tmp/ftmo_dashboard.png')

plot_dashboard(wf_results, ACTIVE_STRATEGY, FTMO_CONFIG)

## 9. 🔍 Overfitting Analysis & Best Params

In [ ]:
def analyse_overfitting(wf_results: dict):
    """Detailed overfitting analysis."""
    fold_results = wf_results['fold_results']

    print('='*70)
    print('🔍 OVERFITTING ANALYSIS')
    print('='*70)

    rows = []
    for r in fold_results:
        ratio = r['oos_is_ratio']
        flag  = '✅ OK' if ratio >= 0.6 else '⚠️  Marginal' if ratio >= 0.3 else '❌ OVERFIT'
        rows.append({
            'Fold'        : r['fold'],
            'IS Sharpe'   : round(r['is_sharpe'],  3),
            'OOS Sharpe'  : round(r['oos_sharpe'], 3),
            'OOS/IS'      : round(ratio, 3),
            'OOS Trades'  : r['oos_metrics'].get('Trades', 0),
            'OOS Win %'   : r['oos_metrics'].get('Win Rate %', 0),
            'OOS MaxDD %' : r['oos_metrics'].get('Max Drawdown %', 0),
            'Assessment'  : flag,
        })

    df_analysis = pd.DataFrame(rows).set_index('Fold')
    print(df_analysis.to_string())

    avg_ratio = np.mean([r['oos_is_ratio'] for r in fold_results])
    overfit_pct = sum(1 for r in fold_results if r['oos_is_ratio'] < 0.3) / len(fold_results) * 100

    print(f'\n📊 Summary:')
    print(f'   Avg OOS/IS ratio : {avg_ratio:.3f}')
    print(f'   Overfit folds    : {overfit_pct:.0f}%')

    if avg_ratio >= 0.6:
        print('   Overall verdict  : ✅ Good OOS generalisation')
    elif avg_ratio >= 0.3:
        print('   Overall verdict  : ⚠️  Moderate — consider simpler parameters')
    else:
        print('   Overall verdict  : ❌ Likely overfit — reduce param complexity')

    # Best params per fold
    print(f'\n🏆 Best Parameters Per Fold:')
    for r in fold_results:
        print(f'   Fold {r["fold"]}: {r["best_params"]}')

    # Param stability analysis
    all_param_keys = list(fold_results[0]['best_params'].keys())
    print(f'\n📐 Parameter Stability (std / range):')
    for pk in all_param_keys:
        vals = [r['best_params'].get(pk, np.nan) for r in fold_results]
        try:
            print(f'   {pk:20s}: min={min(vals):.2f}  max={max(vals):.2f}  '
                  f'std={np.std(vals):.3f}  '
                  f'{"⚠️ UNSTABLE" if np.std(vals) / (max(vals)-min(vals)+1e-9) > 0.4 else "✅ stable"}')
        except:
            print(f'   {pk}: mixed types')

analyse_overfitting(wf_results)

## 10. 🏆 Multi-Strategy Comparison

In [ ]:
def compare_strategies(strategies_to_test: List[str],
                        df: pd.DataFrame,
                        bt_params: dict,
                        n_trials_quick: int = 80) -> pd.DataFrame:
    """
    Run a quick comparison across multiple strategies.
    Uses fewer trials for speed — increase for final selection.
    """
    quick_cfg = OPT_CONFIG.copy()
    quick_cfg['n_trials'] = n_trials_quick
    quick_cfg['n_folds']  = 3  # faster

    summary = []

    for sname in strategies_to_test:
        print(f'\n⚙️  Testing: {sname}')
        sfn   = STRATEGY_REGISTRY[sname]
        sps   = PARAM_SPACES[sname]

        opt = WalkForwardOptimizer(sfn, sps, quick_cfg, backtester)

        try:
            res = opt.optimize(df, bt_params, verbose=False)
        except Exception as e:
            print(f'   ⚠️  Failed: {e}')
            continue

        m = res['agg_metrics']
        summary.append({
            'Strategy'       : sname,
            'OOS Sharpe'     : round(res['avg_oos_sharpe'], 3),
            'Combined Sharpe': round(m.get('Sharpe', 0), 3),
            'Max DD %'       : round(m.get('Max Drawdown %', 0), 2),
            'CAGR %'         : round(m.get('CAGR %', 0), 2),
            'Win Rate %'     : round(m.get('Win Rate %', 0), 1),
            'Profit Factor'  : round(m.get('Profit Factor', 0), 3),
            'Avg OOS/IS'     : round(res['avg_oos_is_ratio'], 3),
            'FTMO PASS'      : m.get('FTMO PASS', False),
        })

    df_comp = pd.DataFrame(summary).sort_values('OOS Sharpe', ascending=False)
    print('\n🏆 STRATEGY COMPARISON:')
    print(df_comp.to_string(index=False))
    return df_comp


# Uncomment to run full comparison:
# comp_results = compare_strategies(
#     list(STRATEGY_REGISTRY.keys()),
#     df_raw, BT_PARAMS,
#     n_trials_quick=80
# )

print('ℹ️  Uncomment the compare_strategies() call above to run multi-strategy comparison.')

## 11. 📋 FTMO Challenge Simulation

In [ ]:
def simulate_ftmo_challenge(
    strategy_name: str,
    best_params: dict,
    df: pd.DataFrame,
    bt_params: dict,
    n_simulations: int = 500,
    challenge_days: int = 30
) -> None:
    """
    Monte Carlo simulation of FTMO challenge outcomes using
    bootstrapped daily returns from the strategy's OOS trades.
    """
    print(f'🎲 Monte Carlo FTMO Challenge Simulation')
    print(f'   Strategy: {strategy_name}  |  Simulations: {n_simulations}  |  Days: {challenge_days}')

    # Get OOS daily returns
    strategy_fn = STRATEGY_REGISTRY[strategy_name]
    sig   = strategy_fn(df, best_params)
    eq, _ = backtester.run(df, sig, **bt_params)
    daily = eq.resample('1D').last().dropna().pct_change().dropna()

    if len(daily) == 0:
        print('⚠️  No daily returns available.')
        return

    account = FTMO_CONFIG['account_size']
    results = {'pass': 0, 'daily_dd_breach': 0, 'total_dd_breach': 0, 'timeout': 0}
    final_equities = []

    np.random.seed(42)
    for _ in range(n_simulations):
        sampled_returns = np.random.choice(daily.values, size=challenge_days, replace=True)
        sim_equity = [account]
        peak = account
        outcome = 'timeout'

        for d, r in enumerate(sampled_returns):
            new_val = sim_equity[-1] * (1 + r)
            sim_equity.append(new_val)
            peak = max(peak, new_val)

            # Daily DD check
            daily_dd = (new_val - sim_equity[-2]) / account
            if daily_dd <= -FTMO_CONFIG['max_daily_dd_pct']:
                outcome = 'daily_dd_breach'
                break

            # Total DD check
            total_dd = (new_val - peak) / peak
            if total_dd <= -FTMO_CONFIG['max_total_dd_pct']:
                outcome = 'total_dd_breach'
                break

            # Profit target check
            if new_val >= account * (1 + FTMO_CONFIG['profit_target_pct']):
                outcome = 'pass'
                break

        results[outcome] += 1
        final_equities.append(sim_equity[-1])

    pass_rate = results['pass'] / n_simulations * 100

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d1117')

    # Outcomes pie
    ax = axes[0]
    ax.set_facecolor('#0d1117')
    labels = ['PASS', 'Daily DD Breach', 'Total DD Breach', 'Timeout (Incomplete)']
    sizes  = [results['pass'], results['daily_dd_breach'],
               results['total_dd_breach'], results['timeout']]
    colors_pie = [FTMO_GREEN, FTMO_RED, '#FF8800', FTMO_BLUE]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'color': 'white', 'fontsize': 9}
    )
    ax.set_title(f'FTMO Challenge Outcomes\nPass Rate: {pass_rate:.1f}%',
                 color='white', fontsize=12, fontweight='bold')

    # Final equity distribution
    ax2 = axes[1]
    ax2.set_facecolor('#0d1117')
    ax2.hist(final_equities, bins=50, color=FTMO_BLUE, alpha=0.7, edgecolor='none')
    ax2.axvline(account * 1.10, color=FTMO_GREEN, ls='--', lw=2, label='10% Target')
    ax2.axvline(account * 0.90, color=FTMO_RED,   ls='--', lw=2, label='10% Max DD')
    ax2.axvline(np.median(final_equities), color=FTMO_GOLD, ls='-', lw=2,
                label=f'Median: ${np.median(final_equities):,.0f}')
    ax2.set_title('Final Equity Distribution', color='white', fontsize=12)
    ax2.set_xlabel('Final Equity ($)', color='white')
    ax2.legend(fontsize=9)
    ax2.tick_params(colors='white')
    ax2.grid(True, alpha=0.3, color='#333355')
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

    plt.tight_layout()
    plt.show()

    print(f'\n📊 Challenge Simulation Results ({n_simulations} runs):')
    print(f'   ✅ PASS              : {results["pass"]:4d}  ({pass_rate:.1f}%)')
    print(f'   ❌ Daily DD Breach   : {results["daily_dd_breach"]:4d}  ({results["daily_dd_breach"]/n_simulations*100:.1f}%)')
    print(f'   ❌ Total DD Breach   : {results["total_dd_breach"]:4d}  ({results["total_dd_breach"]/n_simulations*100:.1f}%)')
    print(f'   ⏰ Timeout           : {results["timeout"]:4d}  ({results["timeout"]/n_simulations*100:.1f}%)')


# Use best params from last fold of walk-forward
best_fold_params = wf_results['fold_results'][-1]['best_params']

simulate_ftmo_challenge(
    strategy_name  = ACTIVE_STRATEGY,
    best_params    = best_fold_params,
    df             = df_raw,
    bt_params      = BT_PARAMS,
    n_simulations  = 500,
    challenge_days = 30,
)

## 12. 💾 Export Results

In [ ]:
def export_results(wf_results: dict, strategy_name: str, prefix: str = 'ftmo_results'):
    """Export all results to CSV/JSON."""
    # Equity curve
    wf_results['combined_equity'].to_csv(f'{prefix}_equity.csv', header=['equity'])

    # Aggregate metrics
    pd.Series(wf_results['agg_metrics']).to_csv(f'{prefix}_metrics.csv', header=['value'])

    # Per-fold summary
    fold_summary = []
    for r in wf_results['fold_results']:
        row = {'fold': r['fold'], 'is_sharpe': r['is_sharpe'],
               'oos_sharpe': r['oos_sharpe'], 'oos_is_ratio': r['oos_is_ratio']}
        row.update({f'param_{k}': v for k, v in r['best_params'].items()})
        fold_summary.append(row)
    pd.DataFrame(fold_summary).to_csv(f'{prefix}_folds.csv', index=False)

    # Trades
    trades_clean = []
    for t in wf_results['all_trades']:
        trades_clean.append({
            'entry': t.get('entry', 0),
            'exit' : t.get('exit', 0),
            'pnl'  : t.get('pnl', 0),
            'direction': t.get('direction', 0),
            'sl'   : t.get('sl', False),
            'tp'   : t.get('tp', False),
        })
    pd.DataFrame(trades_clean).to_csv(f'{prefix}_trades.csv', index=False)

    print(f'✅ Exported:')
    print(f'   {prefix}_equity.csv')
    print(f'   {prefix}_metrics.csv')
    print(f'   {prefix}_folds.csv')
    print(f'   {prefix}_trades.csv')

export_results(wf_results, ACTIVE_STRATEGY)

## 13. 🛠️ Add Your Own Strategy

```python
# 1. Define your strategy function
def strategy_my_custom(df: pd.DataFrame, params: dict) -> pd.Series:
    # ... compute indicators ...
    signal = pd.Series(0, index=df.index)
    signal[long_condition]  =  1
    signal[short_condition] = -1
    return signal

# 2. Register it
STRATEGY_REGISTRY['My Custom'] = strategy_my_custom

# 3. Define its parameter space
PARAM_SPACES['My Custom'] = {
    'period' : ('int',   5, 50),
    'thresh' : ('float', 0.1, 2.0),
    'mode'   : ('categorical', ['fast', 'slow']),
}

# 4. Set ACTIVE_STRATEGY = 'My Custom' and re-run Section 7
```

---
## ⚠️ Important Notes for FTMO
- **News events**: Avoid trading 30min around major news (NFP, FOMC, CPI)
- **Overnight positions**: Some FTMO accounts restrict overnight holds
- **Consistency rule**: Some FTMO variants require no single day > 25% of total profit
- **Position sizing**: The 2% risk per trade is aggressive — consider 0.5–1% for safety
- **OOS ≠ live**: Even with SR > 1.0 OOS, live performance will differ due to execution slippage, psychology, and regime change
- **Carver's ceiling**: Robert Carver (in your references) notes SR > 1.0 is extremely rare for systematic strategies — treat any SR > 0.8 OOS as genuinely strong